In [ ]:
# ============================================================================
# Drought characterization — Rio Negro basin
# Method: Run Theory (Yevjevich, 1967)
# Index: SPEI-12
# Metrics: Frequency, Duration (median), Severity (median)
# Definition: event starts when SPEI < -1 for >= 2 consecutive months
#             event ends when SPEI >= 0
# Reference: Spinoni et al. (2014), Tabari et al. (2021)
# Scenarios: Historical (1985-2014), SSP126 (2041-2070), SSP585 (2041-2070)
# Models: MPI-ESM1-2-HR, UKESM1-0-LL, GFDL-ESM4, IPSL-CM6A-LR, MRI-ESM2-0
# ============================================================================

import os
import numpy as np
import xarray as xr
import netCDF4 as nc
from pathlib import Path

# --- CONFIGURATION ---
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "rio_negro"

path_in  = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/3.outputs_SPEI_R/4.baseline_106_12ac_OF/Rio_Negro/"
path_out = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/"


# ============================================================================
# FUNCTION: Calculate drought metrics for a single time series
# ============================================================================
def calc_drought_metrics(spei_ts):
    """
    Calculates drought frequency, duration and severity using run theory.

    Parameters:
        spei_ts (np.array): SPEI-12 time series for a single pixel

    Returns:
        frequency (int):         number of drought events
        duration_median (float): median duration across all events (months)
        severity_median (float): median severity across all events (sum of |SPEI|)

    Event definition:
        Start: SPEI < -1 for >= 2 consecutive months
        End:   SPEI >= 0
    """
    n           = len(spei_ts)
    in_event    = False
    consec_neg  = 0
    event_start = None
    durations   = []
    severities  = []

    for t in range(n):
        val = spei_ts[t]

        if np.isnan(val):
            in_event   = False
            consec_neg = 0
            continue

        if not in_event:
            if val < -1:
                consec_neg += 1
                if consec_neg >= 2:
                    in_event    = True
                    event_start = t - 1
            else:
                consec_neg = 0
        else:
            if val >= 0:
                duration = (t - 1) - event_start + 1
                severity = np.sum(np.abs(spei_ts[event_start:t]))
                durations.append(duration)
                severities.append(severity)
                in_event   = False
                consec_neg = 0

    # Handle event that does not close before end of series
    if in_event:
        duration = (n - 1) - event_start + 1
        severity = np.sum(np.abs(spei_ts[event_start:n]))
        durations.append(duration)
        severities.append(severity)

    frequency = len(durations)

    if frequency == 0:
        return frequency, np.nan, np.nan

    return frequency, np.median(durations), np.median(severities)


# ============================================================================
# FUNCTION: Save drought metrics as NetCDF
# ============================================================================
def save_netcdf(freq_grid, dur_grid, sev_grid, lon, lat,
                basin, scenario, model, output_dir):
    """
    Saves drought metrics (frequency, duration, severity) into a single NetCDF.
    Arrays are stored in (lat, lon) order following CF conventions.

    Parameters:
        freq_grid:  2D array (lat x lon) of drought frequency
        dur_grid:   2D array (lat x lon) of median drought duration
        sev_grid:   2D array (lat x lon) of median drought severity
        lon, lat:   coordinate arrays
        basin:      basin name string
        scenario:   scenario name string
        model:      model name string (or 'ensemble')
        output_dir: output directory path
    """
    output_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
    ds_out      = nc.Dataset(output_file, "w", format="NETCDF4")

    # Define dimensions — (lat, lon) following CF conventions
    ds_out.createDimension("lat", len(lat))
    ds_out.createDimension("lon", len(lon))

    # Define coordinate variables
    lat_var       = ds_out.createVariable("lat", "f4", ("lat",))
    lat_var.units = "degrees_north"
    lat_var[:]    = lat

    lon_var       = ds_out.createVariable("lon", "f4", ("lon",))
    lon_var.units = "degrees_east"
    lon_var[:]    = lon

    # Arrays are already (lat, lon) — no transpose needed
    freq_save = np.where(np.isnan(freq_grid), -9999, freq_grid)
    dur_save  = np.where(np.isnan(dur_grid),  -9999, dur_grid)
    sev_save  = np.where(np.isnan(sev_grid),  -9999, sev_grid)

    # Frequency variable
    freq_var           = ds_out.createVariable("frequency", "f4", ("lat", "lon"),
                                                fill_value=-9999)
    freq_var.units     = "count"
    freq_var.long_name = "Number of drought events"
    freq_var[:]        = freq_save

    # Duration variable
    dur_var           = ds_out.createVariable("duration", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    dur_var.units     = "months"
    dur_var.long_name = "Median drought duration"
    dur_var[:]        = dur_save

    # Severity variable
    sev_var           = ds_out.createVariable("severity", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    sev_var.units     = "-"
    sev_var.long_name = "Median drought severity (sum of |SPEI| per event)"
    sev_var[:]        = sev_save

    # Global attributes
    ds_out.description = (f"Drought characteristics — {basin} basin, "
                          f"{scenario}, {model}. "
                          f"Run theory (Yevjevich 1967): "
                          f"start SPEI < -1 for >= 2 months, end SPEI >= 0.")
    ds_out.index       = "SPEI-12"
    ds_out.basin       = basin
    ds_out.scenario    = scenario
    ds_out.model       = model

    ds_out.close()
    print(f"    Saved: {output_file}")


# ============================================================================
# FUNCTION: Calculate ensemble median across 5 models
# ============================================================================
def calc_ensemble(basin, scenario, models, output_dir):
    """
    Reads individual model NetCDFs and calculates pixel-wise median
    across all models for each metric.

    Saves result as drought_metrics_{basin}_{scenario}_ensemble.nc
    """
    freq_stack = []
    dur_stack  = []
    sev_stack  = []

    for model in models:
        nc_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)

        freq = ds["frequency"].values.astype(float)
        dur  = ds["duration"].values.astype(float)
        sev  = ds["severity"].values.astype(float)

        # Replace fill values with NaN
        freq = np.where(freq == -9999, np.nan, freq)
        dur  = np.where(dur  == -9999, np.nan, dur)
        sev  = np.where(sev  == -9999, np.nan, sev)

        freq_stack.append(freq)
        dur_stack.append(dur)
        sev_stack.append(sev)

        lon = ds["lon"].values
        lat = ds["lat"].values
        ds.close()

    # Stack → shape (5, lat, lon) → pixel-wise median along model axis
    freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
    dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
    sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)

    # Arrays are already (lat, lon) — no transpose needed
    save_netcdf(freq_ensemble, dur_ensemble, sev_ensemble,
                lon, lat, basin, scenario, "ensemble", output_dir)

    # Print ensemble summary
    print(f"\n  Ensemble summary — {scenario}:")
    for name, grid in [("FREQUENCY", freq_ensemble),
                       ("DURATION",  dur_ensemble),
                       ("SEVERITY",  sev_ensemble)]:
        valid = grid[~np.isnan(grid)]
        print(f"    {name}: min={valid.min():.2f}, "
              f"max={valid.max():.2f}, "
              f"median={np.median(valid):.2f}")


# ============================================================================
# MAIN LOOP — SCENARIOS x MODELS
# ============================================================================
for scenario in scenarios:

    out_dir = f"{path_out}{scenario}/"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario.upper()} | BASIN: {basin.upper()}")
    print(f"{'='*60}")

    for model in models:

        print(f"\n  Processing: {model}")

        # Read SPEI-12 NetCDF
        nc_file = f"{path_in}{scenario}/spei12_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)
        spei    = ds["SPEI_12"].values
        lon     = ds["lon"].values
        lat     = ds["lat"].values
        dims    = ds["SPEI_12"].dims
        ds.close()

        # Replace fill values
        spei = np.where(spei == -9999, np.nan, spei)

        # Ensure (lon, lat, time) order
        if dims[0] == "time":
            spei = np.transpose(spei, (1, 2, 0))

        n_lon, n_lat, _ = spei.shape

        # Initialize output grids — shape (n_lon, n_lat)
        freq_grid = np.full((n_lon, n_lat), np.nan)
        dur_grid  = np.full((n_lon, n_lat), np.nan)
        sev_grid  = np.full((n_lon, n_lat), np.nan)

        # Pixel-by-pixel calculation
        for i in range(n_lon):
            for j in range(n_lat):
                ts = spei[i, j, :]
                if np.all(np.isnan(ts)):
                    continue
                (freq_grid[i, j],
                 dur_grid[i, j],
                 sev_grid[i, j]) = calc_drought_metrics(ts)

        # Print model summary
        for name, grid in [("FREQUENCY", freq_grid),
                           ("DURATION",  dur_grid),
                           ("SEVERITY",  sev_grid)]:
            valid = grid[~np.isnan(grid)]
            print(f"    {name}: min={valid.min():.2f}, "
                  f"max={valid.max():.2f}, "
                  f"median={np.median(valid):.2f}")

        # Save individual model NetCDF
        save_netcdf(freq_grid, dur_grid, sev_grid,
                    lon, lat, basin, scenario, model, out_dir)

    # Calculate and save ensemble median
    print(f"\n  Computing ensemble median — {scenario}")
    calc_ensemble(basin, scenario, models, out_dir)


print(f"\n{'='*60}")
print(f"COMPLETED — Rio Negro basin, 5 models, 3 scenarios")
print(f"Output: {path_out}")
print(f"{'='*60}")


SCENARIO: HISTORICAL | BASIN: RIO_NEGRO

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=2.00, max=11.00, median=5.00
    DURATION: min=7.50, max=68.50, median=19.00
    SEVERITY: min=6.75, max=78.76, median=23.08
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/historical/drought_metrics_rio_negro_historical_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=2.00, max=11.00, median=5.00
    DURATION: min=4.00, max=84.50, median=15.00
    SEVERITY: min=5.01, max=95.79, median=16.02
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/historical/drought_metrics_rio_negro_historical_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=2.00, max=11.00, median=7.00
    DURATION: min=8.00, max=60.00, median=13.50
    SEVERITY: min=6.94, max=74.82, median=16.22
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Output

/tmp/ipykernel_714588/1646835153.py:205: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_714588/1646835153.py:206: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_714588/1646835153.py:207: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    FREQUENCY: min=3.00, max=11.00, median=6.00
    DURATION: min=6.00, max=64.00, median=14.50
    SEVERITY: min=4.35, max=73.18, median=15.90
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/ssp126/drought_metrics_rio_negro_ssp126_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=3.00, max=12.00, median=8.00
    DURATION: min=4.50, max=54.00, median=15.00
    SEVERITY: min=5.06, max=57.98, median=17.23
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/ssp126/drought_metrics_rio_negro_ssp126_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=2.00, max=12.00, median=7.00
    DURATION: min=6.00, max=45.00, median=17.00
    SEVERITY: min=4.00, max=56.28, median=18.68
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/ssp126/drought_metrics_rio_negro_ssp126_GFDL-ESM4.nc

  Processing: IPSL-C

/tmp/ipykernel_714588/1646835153.py:205: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_714588/1646835153.py:206: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_714588/1646835153.py:207: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    FREQUENCY: min=3.00, max=11.00, median=7.00
    DURATION: min=8.00, max=82.00, median=20.00
    SEVERITY: min=5.59, max=96.14, median=22.92
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/ssp585/drought_metrics_rio_negro_ssp585_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=1.00, max=12.00, median=5.00
    DURATION: min=4.00, max=355.00, median=26.00
    SEVERITY: min=4.62, max=406.52, median=29.37
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/ssp585/drought_metrics_rio_negro_ssp585_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=2.00, max=12.00, median=7.00
    DURATION: min=9.00, max=149.00, median=19.50
    SEVERITY: min=7.23, max=175.29, median=25.29
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/ssp585/drought_metrics_rio_negro_ssp585_GFDL-ESM4.nc

  Processing: IP

/tmp/ipykernel_714588/1646835153.py:205: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_714588/1646835153.py:206: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_714588/1646835153.py:207: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


In [2]:
# ============================================================================
# Verification — Rio Negro basin
# ============================================================================

import numpy as np
import xarray as xr
from pathlib import Path

path_out  = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-12/2.Outputs/Rio_Negro/"
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0", "ensemble"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "rio_negro"

print("=" * 60)
print("NETCDF VERIFICATION — Rio Negro basin")
print("=" * 60)

all_ok = True

for scenario in scenarios:
    print(f"\n--- {scenario.upper()} ---")
    for model in models:
        nc_file = Path(f"{path_out}{scenario}/drought_metrics_{basin}_{scenario}_{model}.nc")

        if not nc_file.exists():
            print(f"  ❌ MISSING: {nc_file.name}")
            all_ok = False
            continue

        ds   = xr.open_dataset(nc_file)
        freq = np.where(ds["frequency"].values == -9999, np.nan, ds["frequency"].values.astype(float))
        dur  = np.where(ds["duration"].values  == -9999, np.nan, ds["duration"].values.astype(float))
        sev  = np.where(ds["severity"].values  == -9999, np.nan, ds["severity"].values.astype(float))
        lon  = ds["lon"].values
        lat  = ds["lat"].values
        ds.close()

        n_valid   = np.sum(~np.isnan(freq))
        shape_ok  = freq.ndim == 2
        pixels_ok = (np.sum(~np.isnan(freq)) == np.sum(~np.isnan(dur)) == np.sum(~np.isnan(sev)))
        values_ok = (np.all(freq[~np.isnan(freq)] >= 0) and
                     np.all(dur[~np.isnan(dur)]   >= 0) and
                     np.all(sev[~np.isnan(sev)]   >= 0))
        no_inf    = not (np.any(np.isinf(freq)) or np.any(np.isinf(dur)) or np.any(np.isinf(sev)))

        status = "✅" if (shape_ok and pixels_ok and values_ok and no_inf) else "❌"
        if status == "❌":
            all_ok = False

        print(f"  {status} {model}")
        print(f"       Shape: {freq.shape} | Valid pixels: {n_valid} | "
              f"Freq=[{freq[~np.isnan(freq)].min():.1f},{freq[~np.isnan(freq)].max():.1f}] | "
              f"Dur=[{dur[~np.isnan(dur)].min():.1f},{dur[~np.isnan(dur)].max():.1f}] | "
              f"Sev=[{sev[~np.isnan(sev)].min():.2f},{sev[~np.isnan(sev)].max():.2f}]")

print(f"\n{'='*60}")
print("✅ ALL FILES VERIFIED" if all_ok else "❌ SOME FILES HAVE ISSUES")
print(f"{'='*60}")

NETCDF VERIFICATION — Rio Negro basin

--- HISTORICAL ---
  ✅ MPI-ESM1-2-HR
       Shape: (55, 37) | Valid pixels: 933 | Freq=[2.0,11.0] | Dur=[7.5,68.5] | Sev=[6.75,78.76]
  ✅ UKESM1-0-LL
       Shape: (55, 37) | Valid pixels: 933 | Freq=[2.0,11.0] | Dur=[4.0,84.5] | Sev=[5.01,95.79]
  ✅ GFDL-ESM4
       Shape: (55, 37) | Valid pixels: 933 | Freq=[2.0,11.0] | Dur=[8.0,60.0] | Sev=[6.94,74.82]
  ✅ IPSL-CM6A-LR
       Shape: (55, 37) | Valid pixels: 933 | Freq=[3.0,11.0] | Dur=[6.0,58.0] | Sev=[4.76,45.30]
  ✅ MRI-ESM2-0
       Shape: (55, 37) | Valid pixels: 933 | Freq=[3.0,12.0] | Dur=[7.0,44.0] | Sev=[6.60,50.05]
  ✅ ensemble
       Shape: (55, 37) | Valid pixels: 933 | Freq=[4.0,9.0] | Dur=[10.0,27.0] | Sev=[8.49,31.51]

--- SSP126 ---
  ✅ MPI-ESM1-2-HR
       Shape: (55, 37) | Valid pixels: 933 | Freq=[3.0,11.0] | Dur=[6.0,64.0] | Sev=[4.35,73.18]
  ✅ UKESM1-0-LL
       Shape: (55, 37) | Valid pixels: 933 | Freq=[3.0,12.0] | Dur=[4.5,54.0] | Sev=[5.06,57.98]
  ✅ GFDL-ESM4
       Sh